In [11]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
from scipy.stats import norm
import copy
import os

#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import find_last_thurs_date_of_month, prev_weekday, \
                                                     date_of_prev_thurs, prev_workday_if_holiday, \
                                                     next_working_day_incl_input_date, next_weekday, \
                                                     next_weekday_incl_input_date, get_all_monthly_option_expiries


#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call, BSM_put

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data, get_mthly_df_name_from_expiry, \
                                                     find_latest_available_mibor

#Path Settings
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


In [15]:
# The following functions are user-defined functions used for 
# generating implied vol surface using Brent's Alogrithm

#Error Function - For European call and European put options
#Input: mkt_price of the option, option_type: "CE" for European Call, "PE" for European Put
#      (s_t, k, r, dt) -> (current stock price, strike, risk-free rate, time to maturity)
#Output: Error of model price in comparison with mkt price
def error_func(mkt_price, s_t, k, r, dt, option_type, iv):
    if (option_type == "CE"):
        model_price = BSM_call(s_t, k, r, iv, dt)
    else:
        model_price = BSM_put(s_t, k, r, iv, dt)
    error = model_price - mkt_price
    return error

#### O(n) Algorithm to find product of other elements ####
# Input: a list
#Output: A list with product of other elements except the element in the index
def product_other_elem(arr):
    n = len(arr)
    prod_arr = list(np.ones(n).astype(np.float64))
    
    temp = 1.0
    for i in range(n):
        prod_arr[i] = temp
        temp = temp * arr[i]
    temp = 1.0
    for i in range(n-1, -1, -1):
        prod_arr[i] = prod_arr[i] * temp
        temp = temp * arr[i]
    return prod_arr

#Generates the iv(k) as function of iv(k-1), iv(k-2), iv(k-3), mkt_price, BS price
#### Using Brent's Algorithm ####

#Inverse Quadratic interpolation
def inverse_quadratic_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    g_k = [g(iv) for iv in iv_list]
    g_num = product_other_elem(g_k)
    g_den = [np.prod(g_k[i] - np.array(g_k[:i] + g_k[i+1:])) for i in range(0, len(iv_list))]
    next_iv = np.sum(np.array([(iv_list[i] * g_num[i]) / g_den[i] for i in range(0, len(iv_list))]))
    return next_iv

#Secant method
def secant_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    g_k = [g(iv) for iv in iv_list]
    next_iv = (iv_list[1] - g_k[1]) * ((iv_list[1] - iv_list[0])/(g_k[1]-g_k[0]))
    return next_iv

#Bisection method
def bisection_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    next_iv = (iv_list[0] + iv_list[1])/2
    return next_iv

#Swap the elements of 2 elements list
def swap(iv_list):
    temp = iv_list[1]
    iv_list[1] = iv_list[0]
    iv_list[0] = temp
    return iv_list

#Generate IV using Brent's algorithm
def gen_brents_iv(mkt_price, s_t, k, r, dt, option_type, iv_list_initial):
    iv_list = copy.deepcopy(iv_list_initial)
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    
#     print(g(iv_list[0]), g(iv_list[1]))
    
    try:
        assert (g(iv_list[0]) * g(iv_list[1])) < 0
    except AssertionError:
        print("BSMVol0_price - mkt_price, BSMVol1_price - mkt_price: ", g(iv_list[0]), g(iv_list[1]))
        print("Trade_terms: s_t, k, dt, option_type, iv_list: (", s_t, k, dt, option_type, iv_list, ")")
        print("Mkt Price: ", mkt_price)
        if (option_type == "CE"):
            print("IV0 BSM price: ", BSM_call(s_t, k, r, iv_list[0], dt))
            print("IV1 BSM price: ", BSM_call(s_t, k, r, iv_list[1], dt))
        else:
            print("IV0 BSM price: ", BSM_call(s_t, k, r, iv_list[0], dt))
            print("IV1 BSM price: ", BSM_call(s_t, k, r, iv_list[1], dt))
        print("The initialisations doesn't contain the solution within for interpolation")
    
    if (abs(iv_list[0] ) < abs(iv_list[1])):
        iv_list = swap(iv_list)

    mflag = 1
    
    opt_iv = iv_list[1]    
    num_iter = 0 #maximum iteration is set as 1000 in the while loop below
    eps = 0.0001 # 1 basis point is set as the maximum error allowed for vol

#     print("Starting Error: ", g(opt_iv))
#    error is difference of market and model price
    
    while (abs(g(opt_iv)) >= 0.000001 and num_iter <=1000):   
        if (g(iv_list[0]) != g(iv_list[2]) and g(iv_list[1]) != g(iv_list[2])):
            #Inverse Quadratic Interpolation
            opt_iv = inverse_quadratic_iv(mkt_price, s_t, k, r, dt, option_type, iv_list)
        else:
            #Secant Method
            opt_iv = secant_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list)
        
        if ((( 3 * iv_list[0] + iv_list[1])/4 < opt_iv < iv_list[1]) \
            or ( mflag == 1 and (abs(opt_iv - iv_list[1]) >= abs(iv_list[1] - iv_list[2])/2 )) \
            or ( mflag == 0 and (abs(opt_iv - iv_list[1]) >= abs(iv_list[2] - prev_iv_2)/2 )) \
            or (mflag == 1 and (abs(iv_list[1] - iv_list[2]) < abs(eps))) \
            or (mflag == 0 and (abs(iv_list[2] - prev_iv_2) < abs(eps)))):
            
            opt_iv = (iv_list[0] + iv_list[1]) / 2
            mflag = 1
        else:
            mflag = 0
            
#         print("Iteration",num_iter, ": ", "Price Difference is ", g(opt_iv))
#         print("Model IV: ",  opt_iv)

        prev_iv_2 = iv_list[2]
        iv_list[2] = iv_list[1]
        
        if (g(iv_list[0]) * g(opt_iv) < 0):
            iv_list[1] = opt_iv
        else:
            iv_list[0] = opt_iv
        
        if (abs(iv_list[0] ) < abs(iv_list[1])):
            iv_list = swap(iv_list)
        
        num_iter = num_iter + 1
    return opt_iv

#Function to generate Implied volatility Surface based on Brent's Algorithm
#Note: In vol surface: x-axis is moneyness ie. S/K for call options and K/S for put options
#                      y-axis is Time to maturity
#                      z-axis is implied volatility
#Inputs:
#       dt_list = [tenor1, tenor2....]
#       price_lists_tenorwise = [[Price for all options in tenor 1],[for tenor2],....]
#       strike_lists_tenorwise = [[Strike for all options in tenor 1],[for tenor2],....]
#       s_t - current stock price, r - risk free rate, option_type is "CE" or "PE"
#      initialisation of brent's iv is: [0.0000000001, 50, 50], 
#                 S.T, iv[0] creates a +ve error and iv[1] creates negative error,
#                where, error = (model price - mkt_price)
#Output:
#      A list of iv surface: [dict for tenor 1, dict for tenor 2,...]
#                      dict = {"Moneyness:" [], "T": [list of same value dt], "Impl_Vol": []}
def gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type, iv_list=[0.0000000001, 50, 50]):
    iv_surface = []
    for i in range(0, len(dt_list)):
        dt = dt_list[i]
        strike_list = strike_lists_tenorwise[i]
        price_list = price_lists_tenorwise[i]
        if(option_type == "CE"):
            moneyness_list = [s_t/strike for strike in strike_list]
        else:
            moneyness_list = [strike/s_t for strike in strike_list]
        dt_list_curr_tenor = [dt for evry_elm in price_list]
        impl_vol_list = [gen_brents_iv(price_list[i], s_t, strike_list[i], r, dt, option_type, iv_list) for i in range(0, len(price_list))]
        vol_smile_dict = {"Moneyness":moneyness_list,"T":dt_list_curr_tenor, "Impl_Vol":impl_vol_list}
        iv_surface.append(vol_smile_dict)
    return iv_surface

#Function to generate iv_surface dataframe from (iv-surface list -> list of iv_smile dictionaries)
def gen_iv_surf_dataframe(iv_surface_list_of_dict):
    list_of_df = [pd.DataFrame(iv_surf_dict) for iv_surf_dict in iv_surface_list_of_dict]
    list_of_df = [iv_df.sort_values('Moneyness') for iv_df in list_of_df]
    df_iv_surface = pd.concat(list_of_df, axis=0)
    return df_iv_surface

# We generate and store the volatility surface for 26th March expiry.
# Each day, we generate a volatility surface with time to maturity = 26th march - current date
# ACT/365 day count convention is used
#Input: periods_of =_interest as discussed above, holidays list, product type - "CE" or "PE"
#Output: List of vol surface dataframes and saved as csv in the data location
def gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest, holidays_list, input_path, output_path, stock_ident, prod_type = "CE"):
    expiries_of_interest = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
    #Product Type corresponds to product type we are hedging - theoretically vol smile is same for call and put
    # Put call parity to be tested and volatility arbitrage to be tested at all dates - as analysis of this exercise
    df_mkt = load_all_mthly_data(expiries_of_interest, prod_type_lists=[prod_type], relative_path = input_path, stock_ident=stock_ident)
    df_futures_mkt = load_all_mthly_data(expiries_of_interest, prod_type_lists=["FUT"], relative_path = relative_path, stock_ident=stock_ident)
    df_mibor_hist = pd.read_csv(relative_path + "mibor.csv")
    mthly_list_of_vol_smiles = {}
    for expiry_date in expiries_of_interest:
        fixed_expiry_volsmiles_list = []
        start_date = date_of_prev_thurs(expiry_date, 4)
        curr_date =start_date
        df = df_mkt[get_mthly_df_name_from_expiry(expiry_date, prod_type, holidays_list)]
        df_futures = df_futures_mkt[get_mthly_df_name_from_expiry(expiry_date, "FUT", holidays_list)]
        while(curr_date < expiry_date):
            dt = float((expiry_date - curr_date).days) / 365
            df_curr = df[df['Date'] == curr_date.strftime("%d-%b-%Y")]
            s_t = float(df_curr['Underlying Value'].tolist()[0])
            price_lists_tenorwise = [df_curr['Settle Price'].tolist()]
            strike_lists_tenorwise = [df_curr['Strike Price'].tolist()]
            dt_list = np.array([dt])
#Initially, we had risk free interest rate from futures - commented below - Now from MIBOR
            df_curr_futures = df_futures[df_futures['Date'] == curr_date.strftime("%d-%b-%Y")]
            F_t = float(df_curr_futures['Settle Price'].tolist()[0])
            r = float(1 / dt) * np.log(F_t / s_t)
# Note: The historical data fro mibor file is named as mibor.csv
#             r = find_latest_available_mibor(df_mibor_hist, curr_date)
        
            dict_vol_smile = gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type=prod_type, iv_list=[0.0000000001, 50, 50])[0]
            dict_vol_smile['Date'] = [curr_date.strftime("%d-%b-%Y") for i in range(0, len(dict_vol_smile['Moneyness']))]
            fixed_expiry_volsmiles_list.append(pd.DataFrame(dict_vol_smile))
            curr_date = next_weekday(curr_date)
            curr_date = next_working_day_incl_input_date(curr_date, holidays_list)
        df_fixed_expiry_vol_surface = pd.concat(fixed_expiry_volsmiles_list, axis=0)
        df_fixed_expiry_vol_surface.sort_values(by=['Date', 'Moneyness'])
        mthly_list_of_vol_smiles["Daily_vol_smile_" + prod_type +"_" + expiry_date.strftime("%d-%b-%Y")] = df_fixed_expiry_vol_surface
    for key in mthly_list_of_vol_smiles.keys():
        df = pd.DataFrame(mthly_list_of_vol_smiles[key])
        df.to_csv(output_path + key + ".csv", index = False)
    return mthly_list_of_vol_smiles 



In [21]:
#Index of interest
stock_ident = "BANKNIFTY"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List oh holidays
holidays_list = [date(2020, 3, 10)]


In [17]:

#Generate Daily Vol smile - Time axis for each date 
#correspond to (Expiry date of monthly option - Current date) with ACT/365 Day count convention
gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest={2020: [3]},holidays_list=holidays_list, prod_type = "CE", relative_path = "mkt_data_covid_region/")


FileNotFoundError: [Errno 2] File b'mkt_data_covid_region/OPTIDX_BANKNIFTY_CE_27-Feb-2020_TO_26-Mar-2020.csv' does not exist: b'mkt_data_covid_region/OPTIDX_BANKNIFTY_CE_27-Feb-2020_TO_26-Mar-2020.csv'

/home/jupyter-partha/Vikranth - Chapter 2/Input Data/mkt_data_covid_region/
